In [ ]:
from ultralytics import YOLO
from pathlib import Path
import torch
import os
import shutil
from os.path import exists

Dataset = Path('/content/mitosis_dataset')
Dataset_yaml = Dataset/'data.yaml'

project_name = 'BioSciencia'
model_ver = 'yolov8n'
Trained_model = Path('/content/BioSciencia/weights/best.pt')

image_size = 1280
Batch = 8
Epochs = 300
confidence = 0.25

if torch.cuda.is_available():
  print('GPU Detected...')
  Device = 0
else:
  print('No GPU Detected...')
  print('Training will commence on CPU')
  Device = 'cpu'

if Trained_model.exists():
  print("Pre trained model exists... so training is cancelled")
  model = YOLO(str(Trained_model))
  Train = False
else:
  print('No pre trained model found')
  print('Training a new model')
  model = YOLO(model_ver)
  Train = True

print('Authorizing the dataset')

folder = [Dataset/'train/images' , Dataset/'train/labels',
          Dataset/'valid/images' , Dataset/'valid/labels',
          Dataset/'test/images' , Dataset/'test/labels']
for fld in folder:
  if not fld.exists():
    raise FileNotFoundError(f'{fld} not found')
print('Dataset Authorized')

if Train:
  print('-' * 50)
  print('Initiating the training of new model')
  print('-' * 50)
  results = model.train(data = str(Dataset_yaml), imgsz = image_size, batch = Batch, epochs = Epochs, device = Device, project = 'path',
                        name = project_name, exist_ok = True, save = True, val = True, patience = 50, pretrained = True, workers = 2, verbose = True)
  print('-' * 50)
  print('Training complete')
  print('-' * 50)

else:
  print('-' * 50)
  print('Using pre trained model')
  print('-' * 50)

print('-' * 50)
print('Validating the model')
print('-' * 50)

metrics = model.val(source = str(Dataset_yaml), imgsz = image_size, batch = Batch, device = Device)
print('mAP50: ',round(float(metrics.box.map50),4))
print('mAP50-95: ',round(float(metrics.box.map),4))
print("Precision:", round(float(metrics.box.mp), 4))
print("Recall:", round(float(metrics.box.mr), 4))